In [1]:
import torch
import torch.nn.functional as F

from torch_geometric.nn import GATConv
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader
from torch_geometric.nn.pool import global_mean_pool

from datasets import CVFGATGeometricDataset

In [2]:
device = "cuda"
batch_size = 64

In [3]:
dataset = CVFGATGeometricDataset(
    device, dataset="complete_graph_n5", program="graph_coloring"
)  # list of Data objects

In [4]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [5]:
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=1):
        super().__init__()
        # First GAT layer
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        # Second GAT layer
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        index = (
            torch.LongTensor([[i] * dataset[0].num_nodes for i in range(batch_size)])
            .to(device)
            .flatten()
        )[:x.shape[0]]
        return global_mean_pool(x, index).to(device)


# Model, optimizer, loss
model = GAT(
    in_channels=dataset.num_node_features,
    hidden_channels=8,
    out_channels=1,
    heads=8,
)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

In [6]:
# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    total_loss = torch.FloatTensor([0.0]).to(device=device)
    for batch in loader:
        out = model(batch.x, batch.edge_index)
        loss = F.mse_loss(out.flatten(), batch.y)
        total_loss += loss
        loss.backward()
        optimizer.step()
    return total_loss.item()


# # Testing
def test():
    model.eval()
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    correct = torch.FloatTensor([0.0]).to(device)
    for batch in test_loader:
        out = model(batch.x, batch.edge_index)
        pred = torch.round(out)
        correct += (pred == batch.y).sum()
    acc = int(correct) // len(test_dataset)
    return acc


for epoch in range(1, 101):
    loss = train()
    test_acc = test()
    if epoch % 5 == 0:
        # print(
        #     f"Epoch {epoch:03d}, Loss: {loss:.4f}, Train: {train_acc:.4f}, Val: {val_acc:.4f}, Test: {test_acc:.4f}"
        # )
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Test: {test_acc:.2f}")

Epoch 005, Loss: 140.3717, Test: 0.00
Epoch 010, Loss: 211.2442, Test: 4.00
Epoch 015, Loss: 51.1620, Test: 28.00
Epoch 020, Loss: 20.7148, Test: 29.00
Epoch 025, Loss: 20.2171, Test: 28.00
Epoch 030, Loss: 18.3784, Test: 28.00
Epoch 035, Loss: 18.2352, Test: 28.00
Epoch 040, Loss: 17.6138, Test: 26.00
Epoch 045, Loss: 28.8596, Test: 29.00
Epoch 050, Loss: 17.3203, Test: 27.00
Epoch 055, Loss: 16.6782, Test: 28.00
Epoch 060, Loss: 20.9405, Test: 26.00
Epoch 065, Loss: 36.9789, Test: 27.00
Epoch 070, Loss: 62.5318, Test: 17.00
Epoch 075, Loss: 19.8010, Test: 25.00
Epoch 080, Loss: 39.9607, Test: 17.00
Epoch 085, Loss: 48.0565, Test: 17.00
Epoch 090, Loss: 101.1120, Test: 25.00
Epoch 095, Loss: 148.8861, Test: 1.00
Epoch 100, Loss: 21.2594, Test: 22.00
